# YAM Peg-in-Hole — Kaggle / Colab ACT Training (`dream0906/yam-v3`)

在 Kaggle 免费 GPU（T4 x2）或 Colab T4 上，基于 Hugging Face 数据集 `dream0906/yam-v3`（100% 纯净直插示教）训练 ACT 策略。

**Kaggle 设置**
1. Notebook → Settings → Accelerator：`GPU T4 x2`
2. Internet：`On`（首次运行需下载数据集和依赖）
3. 单次会话上限 12 小时；检查点每 5000 步写入 `outputs/train/act_yam_peg_v3`
4. 训练完成后在右侧 **Output** 面板下载 `act_yam_peg_pretrained.zip`

In [ ]:
!nvidia-smi
import os
import sys

import torch

print("python", sys.version.split()[0])
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(), "| gpus", torch.cuda.device_count())
print("cwd", os.getcwd())

In [ ]:
%%bash
pip install -q uv
uv venv --clear /kaggle/working/venv --python 3.12
uv pip install --python /kaggle/working/venv/bin/python -q "lerobot[dataset,training]"
/kaggle/working/venv/bin/python -c "import torch, lerobot; print('lerobot', lerobot.__version__, '| cuda', torch.cuda.is_available())"

In [ ]:
%%bash
/kaggle/working/venv/bin/python - << 'EOF'
from lerobot.datasets.lerobot_dataset import LeRobotDatasetMetadata

HF_DATASET_ID = "dream0906/yam-v3"
meta = LeRobotDatasetMetadata(HF_DATASET_ID)
print("FPS:", meta.fps)
print("episodes:", meta.total_episodes, "frames:", meta.total_frames)
print("features:", list(meta.features.keys()))
EOF

In [ ]:
%%bash
# 与 v1 完全同配置（batch 8 / 30k 步 / chunk 100），唯一变量是场景几何
PYTHONUNBUFFERED=1 /kaggle/working/venv/bin/lerobot-train \
  --dataset.repo_id=dream0906/yam-v3 \
  --policy.type=act \
  --policy.device=cuda \
  --batch_size=8 \
  --num_workers=2 \
  --steps=30000 \
  --save_freq=5000 \
  --log_freq=200 \
  --policy.push_to_hub=false \
  --output_dir=outputs/train/act_yam_peg_v3

In [ ]:
# 可选：会话中断后从最近检查点续训
# %%bash
# PYTHONUNBUFFERED=1 /kaggle/working/venv/bin/lerobot-train --resume=true \
#   --config_path=outputs/train/act_yam_peg_v3/checkpoints/last/pretrained_model/train_config.json

In [ ]:
!zip -r -q act_yam_peg_pretrained.zip outputs/train/act_yam_peg_v3/checkpoints/last/pretrained_model
!ls -lh act_yam_peg_pretrained.zip
print("Kaggle: 右侧 Output 面板下载；Colab: 左侧文件栏下载")

**备注**

- 若视频解码报 torchcodec / ffmpeg 错误，在训练命令追加 `--dataset.video_backend=pyav`。
- Kaggle 免费 GPU 每周约 30 小时额度；可先 `--steps=2000` 确认 loss 正常下降，再跑满 30000 步。
- 产物只包含 `pretrained_model/`（config、权重、processor）；下载到本地后运行：
  `python scripts/eval_policy.py --checkpoint <...>/pretrained_model --episodes 50 --te-coeff 0.01`